# Tree Removal — keep all 37 pole structures

**One job:** take `poles_with_crossarms_qgis.las` (64 clusters, ~148k points) and remove every point belonging to a Tree cluster. Everything else — 33 Electrical Poles, 2 generic Poles, 2 Streetlights, 37 structures total — stays.

**How it works:** every point in the LAS came from one of the 64 clusters defined in `pole_locations.csv`. Those clusters were built by expanding each center into a ~2.5m cylinder, and DBSCAN (eps=2m) guarantees centers don't overlap. So for any point in the LAS, the nearest center **is** its cluster of origin. Drop a point iff its nearest center has `Attribute == 'Tree'`.

This is provably lossless for non-tree clusters — no pole point can be closer to a tree center than to its own pole center, because they were cluster-separated to begin with.

**Run this AFTER:**
1. `poles.ipynb` has produced `poles_with_crossarms_qgis.las` + `pole_locations.csv`
2. You've done the manual KML labelling in Google Earth and added the `Attribute` column to `pole_locations.csv`

**Run this BEFORE:** anything else (component labelling, wires, etc.). This is Phase 1 step 1.

In [1]:
# Step 1 — Load inputs and show what we're working with

import numpy as np
import pandas as pd
import laspy
from scipy.spatial import cKDTree

LAS_IN  = 'poles_with_crossarms_qgis1.las'
CSV_IN  = 'pole_locations.csv'
LAS_OUT = 'poles_no_trees.las'
CSV_OUT = 'pole_locations_no_trees.csv'

# --- Load the 64-cluster CSV ---
df = pd.read_csv(CSV_IN)
# Inject user's manual labels from backup latlon CSV
_backup = pd.read_csv('poles_latlon.csv')
df['Attribute'] = df['Pole_ID'].map(_backup.set_index('Pole_ID')['Attribute'])
print(f'CSV: {len(df)} cluster centers loaded')

assert 'Attribute' in df.columns, (
    "pole_locations.csv has no 'Attribute' column. "
    "Go back to Google Earth, label each point, export the KML back to CSV, and merge the Attribute column."
)

# Clean up any stray whitespace from Google Earth exports
df['Attribute'] = df['Attribute'].astype(str).str.strip()

print('\nAttribute breakdown in CSV:')
for name, count in df['Attribute'].value_counts().items():
    marker = '  DROP' if name == 'Tree' else '  KEEP'
    print(f'  {name:<20} {count:>3}  ({marker})')

n_trees = (df['Attribute'] == 'Tree').sum()
n_keep  = len(df) - n_trees
print(f'\nTarget: keep {n_keep} structures, drop {n_trees} trees')

# --- Load the LAS ---
las = laspy.read(LAS_IN)
pts = np.vstack([las.x, las.y, las.z]).T
print(f'\nLAS: {len(pts):,} points loaded')

CSV: 64 cluster centers loaded

Attribute breakdown in CSV:
  Electrical Pole       33  (  KEEP)
  Tree                  27  (  DROP)
  Streetlight            2  (  KEEP)
  Pole                   2  (  KEEP)

Target: keep 37 structures, drop 27 trees

LAS: 134,899 points loaded


In [2]:
# Step 2 — Assign each point to its nearest cluster center, count by Attribute

centers = df[['X', 'Y']].values.astype(float)   # shape (64, 2)
tree = cKDTree(centers)

# For each point, find the single nearest center index (k=1)
_, nearest_idx = tree.query(pts[:, :2], k=1)

# Attach the Attribute label to each point via its cluster index
attr_per_cluster = df['Attribute'].values
attr_per_point = attr_per_cluster[nearest_idx]

# Count how many points came from each Attribute type
print('Points per cluster type (before filtering):')
unique_attrs, counts = np.unique(attr_per_point, return_counts=True)
for a, c in zip(unique_attrs, counts):
    marker = '  DROP' if a == 'Tree' else '  KEEP'
    print(f'  {a:<20} {c:>7,}  ({marker})')

Points per cluster type (before filtering):
  Electrical Pole       65,036  (  KEEP)
  Pole                   2,730  (  KEEP)
  Streetlight            1,800  (  KEEP)
  Tree                  65,333  (  DROP)


In [3]:
# Step 3 — Apply the filter

keep_mask = attr_per_point != 'Tree'
kept_pts = pts[keep_mask]
kept_idx = nearest_idx[keep_mask]           # cluster index for each kept point

print(f'Kept:    {len(kept_pts):>7,} points  ({100.0*keep_mask.mean():.1f}% of input)')
print(f'Dropped: {(~keep_mask).sum():>7,} points  ({100.0*(~keep_mask).mean():.1f}% of input)')

# Sanity check — every kept point's cluster MUST be non-tree
kept_attrs = attr_per_cluster[kept_idx]
assert (kept_attrs != 'Tree').all(), 'BUG: a tree point slipped through the filter'

# Report which of the 37 non-tree clusters contributed points
kept_cluster_ids = np.unique(kept_idx)
print(f'\nClusters preserved: {len(kept_cluster_ids)} / {n_keep} expected')
if len(kept_cluster_ids) < n_keep:
    missing = set(df[df['Attribute'] != 'Tree']['Pole_ID']) - set(kept_cluster_ids)
    print(f'  Missing Pole_IDs (no points assigned): {sorted(missing)}')
    print('  (These clusters had their points captured by another nearer center — rare, investigate if many.)')

Kept:     69,566 points  (51.6% of input)
Dropped:  65,333 points  (48.4% of input)

Clusters preserved: 37 / 37 expected


In [4]:
# Step 4 — Save the filtered LAS, preserving the original header (CRS, scales, offsets)

header = laspy.LasHeader(
    point_format=las.header.point_format,
    version=las.header.version,
)
header.scales  = las.header.scales
header.offsets = las.header.offsets

out_las = laspy.LasData(header)
out_las.x = kept_pts[:, 0]
out_las.y = kept_pts[:, 1]
out_las.z = kept_pts[:, 2]
out_las.write(LAS_OUT)

print(f'Wrote {LAS_OUT} with {len(kept_pts):,} points')

# Verify round-trip
check = laspy.read(LAS_OUT)
assert len(check.x) == len(kept_pts), 'Round-trip mismatch'
print(f'Verified: file has {len(check.x):,} points on disk')

Wrote poles_no_trees.las with 69,566 points
Verified: file has 69,566 points on disk


In [5]:
# Step 5 — Save the filtered CSV with just the 37 non-tree structures

df_clean = df[df['Attribute'] != 'Tree'].reset_index(drop=True)
df_clean.to_csv(CSV_OUT, index=False)

print(f'Wrote {CSV_OUT} with {len(df_clean)} rows')
print('\nAttribute breakdown after filtering:')
for name, count in df_clean['Attribute'].value_counts().items():
    print(f'  {name:<20} {count:>3}')

# NOTE: Pole_ID values in the filtered CSV are the ORIGINAL IDs from pole_locations.csv,
# not renumbered 0-36. This is intentional — it means any masks or references you made
# using the original Pole_IDs still match correctly.

Wrote pole_locations_no_trees.csv with 37 rows

Attribute breakdown after filtering:
  Electrical Pole       33
  Streetlight            2
  Pole                   2


## Done

**Outputs:**
- `poles_no_trees.las` — same as `poles_with_crossarms_qgis.las` but with all tree-cluster points removed. All 37 non-tree structures preserved.
- `pole_locations_no_trees.csv` — 37-row CSV with original Pole_IDs kept.

**Visual check in QGIS:**
1. Load `poles_no_trees.las` alongside the original `poles_with_crossarms_qgis.las`
2. Toggle between them — the tree clumps (the blobby ones at the north end, plus scattered trees along the road) should disappear
3. Every vertical pole-shaped structure should remain

**If any pole is missing:** look at the print output from Step 3 — `Missing Pole_IDs`. That tells you which non-tree clusters lost all their points (should be zero or very few).